In [1]:
import torch
import torch.nn.functional as F
from unsloth import FastLanguageModel
from transformers import AutoModelForCausalLM, AutoTokenizer

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


### Dataset Generation

In [2]:
from datasets import load_dataset

TRAIN_SIZE = 2000
dataset = load_dataset("allenai/tulu-3-sft-mixture", split="train", streaming=True)
dataset.shuffle(seed=42)
train_dataset = []
for i, x in enumerate(dataset):
    if i >= TRAIN_SIZE:
        break
    train_dataset.append(dict(messages=x['messages']))

train_dataset[0]

{'messages': [{'content': 'Create a snippet of Terraform HCL code that create an AWS autoscaling group, and an ALB in front to expose an application to internet.',
   'role': 'user'},
  {'content': 'Sure, here\'s an example Terraform HCL code that creates an AWS Autoscaling Group and an Application Load Balancer to expose an application to the internet:\n``` \n# Configure the AWS provider\nprovider "aws" {\n  region = "us-east-1"\n}\n\n# Create a security group to allow traffic to the ALB\nresource "aws_security_group" "alb_sg" {\n  name_prefix = "alb_sg"\n  ingress {\n    from_port = 80\n    to_port = 80\n    protocol = "tcp"\n    cidr_blocks = ["0.0.0.0/0"]\n  }\n}\n\n# Create an ALB and target group\nresource "aws_lb" "alb" {\n  name               = "example-alb"\n  internal           = false\n  load_balancer_type = "application"\n\n  subnets = ["subnet-12345678", "subnet-87654321"]\n\n  security_groups = [aws_security_group.alb_sg.id]\n\n  tags = {\n    Environment = "production"\n

In [3]:
import os

max_seq_length = 1024
STUDENT_MODEL_NAME = "./knowledge-ingestion-test/model/checkpoint-178"
os.path.exists(STUDENT_MODEL_NAME)


True

In [4]:
from peft import PeftConfig, PeftModel

config = PeftConfig.from_pretrained(STUDENT_MODEL_NAME)
config.base_model_name_or_path

'unsloth/Qwen3-4B-Instruct-2507'

In [5]:
base_model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    torch_dtype=torch.bfloat16,
    device_map="cuda:0",
)
base_model.eval()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560, padding_idx=151654)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
        (

In [6]:
# student_model = base_model
student_model = PeftModel.from_pretrained(
    base_model,
    STUDENT_MODEL_NAME,
    is_trainable=False,   # True if you want to keep training
)
student_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 2560, padding_idx=151654)
        (layers): ModuleList(
          (0-35): 36 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2560, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2560, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_p

In [7]:
for name, param in student_model.named_parameters():
    if "lora" in name:
        param.requires_grad = True
        assert param.requires_grad == True
    else:
        assert param.requires_grad == False



In [8]:
device = next(student_model.parameters()).device
device

device(type='cuda', index=0)

In [9]:
# ----------------------------
# 2) Load frozen teacher
# ----------------------------
# For THIS minimal implementation, assume teacher/student share tokenizer ids
teacher = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    torch_dtype=torch.bfloat16,
    device_map="cuda:1",
)
teacher.eval()

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560, padding_idx=151654)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
        (

In [10]:
tokenizer = AutoTokenizer.from_pretrained(config.base_model_name_or_path)

In [11]:
optimizer = torch.optim.AdamW(student_model.parameters(), lr=1e-5)

In [12]:
@torch.no_grad()
def sample_from_student(messages, max_new_tokens=256, temperature=1.0, top_p=1.0):
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt", padding=True, padding_side="left").to(device)
    prompt_len = inputs["input_ids"].shape[1]

    sampled = student_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    return sampled, prompt_len


sampled, prompt_len = sample_from_student([{'role': 'user', 'content': 'Hello, how are you?'}])
print(tokenizer.decode(sampled[0], skip_special_tokens=True))

user
Hello, how are you?
assistant
<translation group="assistant">Hello, I am an AI assistant developed by Alibaba Cloud, and I can provide you with assistance in multiple languages.</translation>


In [13]:
optimizer.zero_grad()
student_model.zero_grad()

In [14]:

max_grad_norm = 1.0

In [ ]:
# One Step
def opd_step(messages):
    # generate rollout
    sampled, prompt_len = sample_from_student(messages)

    optimizer.zero_grad()
    student_model.zero_grad()

    full_ids = sampled
    full_ids = full_ids.to(device)
    attn = torch.ones_like(full_ids, device=device)
    full_ids.shape, attn.shape

    # get student logprobs
    out = student_model(input_ids=full_ids, attention_mask=attn)
    student_logprobs = F.log_softmax(out.logits, dim=-1)
    student_logprobs.shape

    # get teacher logprobs
    with torch.no_grad():
        teacher_out = teacher(input_ids=full_ids.to(teacher.device), attention_mask=attn.to(teacher.device))
        teacher_logprobs = F.log_softmax(teacher_out.logits, dim=-1)
    teacher_logprobs.shape

    # get advantage
    with torch.no_grad():
        advantage = teacher_logprobs.to(student_logprobs.device) - student_logprobs.detach()
    advantage.shape

    # get loss
    full_loss = advantage * student_logprobs

    # ignore mask
    ignore_mask = torch.ones_like(full_ids)
    ignore_mask[:, :prompt_len] = 0
    ignore_mask.shape
    ignore_mask = ignore_mask.unsqueeze(-1).expand_as(full_loss)
    ignore_mask.shape

    # get loss
    loss = -(full_loss * ignore_mask).sum() / ignore_mask.sum()

    # backprop
    loss.backward()
    total_norm = torch.nn.utils.clip_grad_norm_(student_model.parameters(), 1.0)
    optimizer.step()
    return {
        'loss': loss.item(),
        'total_norm': total_norm.item()
    }

In [ ]:
BATCH_SIZE = 1
import json

for idx in range(0, len(train_dataset), BATCH_SIZE):
    batch = [x['messages'] for x in train_dataset[idx:idx+BATCH_SIZE]]
    stats = opd_step(batch)
    print(json.dumps(stats))


loss: -424.0
{"loss": -424.0, "total_norm": 526.010009765625}
loss: -237.0
{"loss": -237.0, "total_norm": 422.9808349609375}
loss: -312.0
{"loss": -312.0, "total_norm": 379.1833801269531}
loss: -116.0
{"loss": -116.0, "total_norm": 115.94364929199219}
loss: -112.0
{"loss": -112.0, "total_norm": 181.11167907714844}
loss: -430.0
{"loss": -430.0, "total_norm": 1017.9256591796875}
loss: -422.0
{"loss": -422.0, "total_norm": 881.3943481445312}
loss: -150.0
{"loss": -150.0, "total_norm": 335.3515319824219}
loss: -388.0
{"loss": -388.0, "total_norm": 3156.6142578125}
loss: -180.0
{"loss": -180.0, "total_norm": 242.17919921875}
loss: -103.5
{"loss": -103.5, "total_norm": 144.94232177734375}
loss: -201.0
{"loss": -201.0, "total_norm": 224.0335693359375}
loss: -144.0
{"loss": -144.0, "total_norm": 148.2090301513672}
loss: -342.0
{"loss": -342.0, "total_norm": 653.64453125}
loss: -142.0
{"loss": -142.0, "total_norm": 183.25555419921875}
loss: -212.0
{"loss": -212.0, "total_norm": 588.798156738281